# Priority-Tier Queueing in an LLM API Inference Service

**Submitted by:**
- Pelayo, Agatha Fei R.
- Sanchez, Jasmin Ariane C.
- Santos, Lebbeus Ryan V.
- Tazarte, Mico Angelo C.

**Submitted to:**
- Mr. Martin Joseph O. Guinto

**Date:**
- September 30, 2026

## 1. Project Background and Problem Statement
The current system operates a 2-GPU inference service for a Large Language Model (LLM) API, processing requests on a strict First-Come, First-Served (FCFS) basis. During peak hours (8 requests/minute), the system bottlenecks. Because 80% of the traffic comes from Free-tier users, premium Paid-tier users are stuck in the same waiting line, risking a breach of our 100-second maximum Service Level Agreement (SLA).

## 2. Simulation Goal
The objective of this simulation is to evaluate two alternative solutions to resolve the bottleneck and protect the Paid-tier SLA:
- **Alternative 1 (Software Solution):** Implement a Priority Queue to serve Paid-tier users first.
- **Alternative 2 (Hardware Solution):** Purchase a 3rd GPU to increase overall processing capacity while maintaining FCFS.

## 3. Methodology
Using `simpy`, we model the system across 30 independent 24-hour replications. We evaluate the average wait times and SLA breach percentages for both tiers under normal peak conditions, followed by a +20% traffic sensitivity analysis (stress test) to determine system resilience.

**The Concept:** This simulation creates a digital "twin" of a real-world AI server. Instead of testing on live servers and risking actual crashes, we use Python to generate virtual users who send API requests over a 24-hour period. We can safely observe where the system bottlenecks and test different queueing policies (like VIP lanes or adding hardware) to see which one best protects our premium users' Service Level Agreement (SLA) by simulating this traffic.

## Global Parameters & Imports

**The Concept:** Before running the simulation, we need to establish the "laws of physics" for our virtual world. These global variables dictate how the system behaves across all of our tests so that our comparisons between the Baseline and the Alternatives remain 100% fair.

**How the Code Works:**
- **Arrival Rates:** We use a statistical model (Poisson process) to simulate random user traffic, which spikes to `PEAK_ARRIVAL_RATE = 8` requests per minute during busy windows.
- **Service Time:** We use an Exponential distribution to simulate the GPU generating AI text, taking an average of `MEAN_SERVICE_TIME = 11` seconds per request.
- **SLA Threshold:** This is our critical "fail state" boundary. If the queue gets so long that a user waits more than 100 seconds before a GPU becomes available, the simulation flags it as an SLA breach.

In [1]:
!pip install -q simpy

import simpy
import numpy as np
import pandas as pd
import random

print("Libraries successfully imported!")

# Simulation Configurations
NUM_GPUS = 2

# Arrival process (Poisson) - requests per minute
OFF_PEAK_ARRIVAL_RATE = 3
PEAK_ARRIVAL_RATE = 8

# Peak windows, in seconds from midnight (00:00 = 0)
PEAK_WINDOWS = [
    (9 * 3600, 12 * 3600),   # 9:00 AM - 12:00 PM
    (18 * 3600, 21 * 3600),  # 6:00 PM - 9:00 PM
]

# Service process (Exponential) - mean service time in seconds per request
MEAN_SERVICE_TIME = 11

# Tier probabilities
FREE_TIER_PROB = 0.80
PAID_TIER_PROB = 0.20

# SLA performance limit
SLA_THRESHOLD = 100  # seconds

# Experimental replications
NUM_REPLICATIONS = 30
SIM_DURATION = 86400  # 24-hour simulation run

def get_current_arrival_rate(current_time):
    time_of_day = current_time % 86400
    for start, end in PEAK_WINDOWS:
        if start <= time_of_day < end:
            return PEAK_ARRIVAL_RATE
    return OFF_PEAK_ARRIVAL_RATE

Libraries successfully imported!


## Scenario 1: Baseline (FCFS)
**The Concept:** This represents our current struggling system. All incoming API requests (both Free and Paid) are placed into a single waiting line. Whoever arrives first gets processed first, exactly like a standard line at a grocery store checkout.

**How the Code Works:** We use a standard `simpy.Resource` with a capacity of 2 (representing our 2 GPUs). When a user arrives, the code calls `with gpu_resource.request() as req: yield req`. This tells Python to put the user in line. If both GPUs are busy computing text, the code essentially pauses this user's timer until a GPU finishes its previous task and becomes available. Because it is standard FCFS, the code does not check if the user is Free or Paid; it just serves the next person in line.

In [2]:
# ==========================================
# SCENARIO 1: BASELINE (FCFS)
# ==========================================

free_sla_breaches = 0
paid_sla_breaches = 0
total_free_users = 0
total_paid_users = 0

free_wait_times = []
paid_wait_times = []

def request_process_fcfs(env, name, tier, gpu_resource):
    global free_sla_breaches, paid_sla_breaches, total_free_users, total_paid_users

    arrival_time = env.now

    with gpu_resource.request() as req:
        yield req

        wait_time = env.now - arrival_time

        if tier == 'Free':
            total_free_users += 1
            free_wait_times.append(wait_time)
            if wait_time > SLA_THRESHOLD:
                free_sla_breaches += 1
        else:
            total_paid_users += 1
            paid_wait_times.append(wait_time)
            if wait_time > SLA_THRESHOLD:
                paid_sla_breaches += 1

        service_duration = random.expovariate(1.0 / MEAN_SERVICE_TIME)
        yield env.timeout(service_duration)

def arrival_generator_fcfs(env, gpu_resource):
    user_count = 0
    while True:
        current_rate = get_current_arrival_rate(env.now)
        if current_rate == 0:
            yield env.timeout(60)
            continue
        interarrival_time = random.expovariate(current_rate / 60.0)
        yield env.timeout(interarrival_time)

        user_count += 1
        tier = 'Paid' if random.random() < PAID_TIER_PROB else 'Free'
        env.process(request_process_fcfs(env, f"User_{user_count}", tier, gpu_resource))

### Check 1: Zero-Traffic Test (Verification)

Both the off-peak and peak arrival rates are set to **0 requests/minute** for a single 24-hour Baseline run.

**Expected:** 0 requests generated, 0 waits recorded, and both GPU slots idle at the end of the run.

*Note:* A guard was added to `arrival_generator_fcfs` so that a zero arrival rate advances the clock by 60 seconds instead of dividing by zero. This does not affect results at non-zero rates.

In [3]:
# ==========================================
# CHECK 1: ZERO-TRAFFIC TEST (Verification)
# ==========================================
PEAK_ARRIVAL_RATE, OFF_PEAK_ARRIVAL_RATE = 0, 0

random.seed(42)
np.random.seed(42)
env_zero = simpy.Environment()
gpu_zero = simpy.Resource(env_zero, capacity=NUM_GPUS)
env_zero.process(arrival_generator_fcfs(env_zero, gpu_zero))

free_sla_breaches, paid_sla_breaches = 0, 0
total_free_users, total_paid_users = 0, 0
free_wait_times, paid_wait_times = [], []

env_zero.run(until=SIM_DURATION)

n_requests = total_free_users + total_paid_users
print("=== CHECK 1: Zero-Traffic Test ===")
print(f"Requests generated: {n_requests}")
print(f"Waits recorded: {len(free_wait_times) + len(paid_wait_times)}")
print(f"GPUs busy at end: {gpu_zero.count}")
print("PASSED" if n_requests == 0 and gpu_zero.count == 0 else "FAILED")

PEAK_ARRIVAL_RATE, OFF_PEAK_ARRIVAL_RATE = 8, 3   # restore normal rates

=== CHECK 1: Zero-Traffic Test ===
Requests generated: 0
Waits recorded: 0
GPUs busy at end: 0
PASSED


## Scenario 2: Alternative 1 (Priority Queue)
**The Concept:** This represents our software-based solution. Instead of buying new hardware, we implement a "VIP Lane" for our premium users. Paid-tier users are allowed to cut in front of Free-tier users in the waiting line to ensure their 100-second SLA is protected.

**How the Code Works:** We replace the standard resource with a `simpy.PriorityResource`. In SimPy, a **lower priority number means a higher priority in line**. We assign Paid users a priority of `0` and Free users a priority of `1`. When a GPU finishes a task and looks at the queue, it will actively search for any `0`s (Paid users) and pull them out of the line first, even if a `1` (Free user) has been waiting longer.

In [4]:
# ==========================================
# SCENARIO 2: ALTERNATIVE 1 (PRIORITY QUEUE)
# ==========================================

free_sla_breaches_priority = 0
paid_sla_breaches_priority = 0
total_free_users_priority = 0
total_paid_users_priority = 0

free_wait_times_priority = []
paid_wait_times_priority = []

def request_process_priority(env, name, tier, gpu_resource):
    global free_sla_breaches_priority, paid_sla_breaches_priority
    global total_free_users_priority, total_paid_users_priority

    arrival_time = env.now

    # Paid users get priority 0, Free users get priority 1
    priority_val = 0 if tier == 'Paid' else 1

    with gpu_resource.request(priority=priority_val) as req:
        yield req

        wait_time = env.now - arrival_time

        if tier == 'Free':
            total_free_users_priority += 1
            free_wait_times_priority.append(wait_time)
            if wait_time > SLA_THRESHOLD:
                free_sla_breaches_priority += 1
        else:
            total_paid_users_priority += 1
            paid_wait_times_priority.append(wait_time)
            if wait_time > SLA_THRESHOLD:
                paid_sla_breaches_priority += 1

        service_duration = random.expovariate(1.0 / MEAN_SERVICE_TIME)
        yield env.timeout(service_duration)

def arrival_generator_priority(env, gpu_resource):
    user_count = 0
    while True:
        current_rate = get_current_arrival_rate(env.now)
        interarrival_time = random.expovariate(current_rate / 60.0)
        yield env.timeout(interarrival_time)

        user_count += 1
        tier = 'Paid' if random.random() < PAID_TIER_PROB else 'Free'
        env.process(request_process_priority(env, f"User_{user_count}", tier, gpu_resource))

### Check 2: Event Trace (Verification)

One 24-hour run of **Alternative 1 (Priority Queue)** with **seed 42**, with every arrival, service start, and completion logged. Logging is done by wrapping SimPy's `PriorityResource`, so the model's own functions are not modified.

The full run is checked automatically for three conditions:
1. No request begins service while both GPU slots are busy.
2. A waiting paid-tier request is always served before a waiting free-tier request.
3. A request in service is never interrupted (non-preemption).

The first 20 events of the 9:00 AM peak window (t = 32,400 s) are printed for manual inspection, along with one example of a paid request moving ahead of an earlier free request.

**Expected:** 0 violations for all three conditions.

In [5]:
# ==========================================
# CHECK 2: EVENT TRACE (Verification)
# ==========================================
# Logs every arrival, service start, and completion by wrapping SimPy's
# PriorityResource. The notebook's own functions are NOT modified.
PEAK_ARRIVAL_RATE, OFF_PEAK_ARRIVAL_RATE = 8, 3   # make sure normal rates are used

trace = []

class TracedPriorityResource(simpy.PriorityResource):
    def request(self, priority=0, **kwargs):
        req = super().request(priority=priority, **kwargs)
        tier = "Paid" if priority == 0 else "Free"
        req.rid = len([e for e in trace if e[3] == "ARRIVE"]) + 1
        trace.append((self._env.now, req.rid, tier, "ARRIVE", len(self.queue), self.count, None))

        def on_start(event, tier=tier, rid=req.rid):
            paid_waiting = sum(1 for r in self.queue if r.priority == 0)
            trace.append((self._env.now, rid, tier, "START", len(self.queue), self.count, paid_waiting))

        req.callbacks.append(on_start)
        return req

    def release(self, request):
        result = super().release(request)
        tier = "Paid" if request.priority == 0 else "Free"
        trace.append((self._env.now, request.rid, tier, "DONE", len(self.queue), self.count, None))
        return result

# One 24-hour run of Alternative 1 with seed 42
random.seed(42)
np.random.seed(42)
env_trace = simpy.Environment()
gpu_trace = TracedPriorityResource(env_trace, capacity=NUM_GPUS)
env_trace.process(arrival_generator_priority(env_trace, gpu_trace))

free_sla_breaches_priority, paid_sla_breaches_priority = 0, 0
total_free_users_priority, total_paid_users_priority = 0, 0
free_wait_times_priority, paid_wait_times_priority = [], []

env_trace.run(until=SIM_DURATION)

starts = [e for e in trace if e[3] == "START"]
over_capacity = sum(1 for e in starts if e[5] > NUM_GPUS)
priority_violations = sum(1 for e in starts if e[2] == "Free" and e[6] > 0)
restarted = len(starts) - len({e[1] for e in starts})

print("=== CHECK 2: Event Trace (Alternative 1, seed 42) ===")
print(f"Resource type: PriorityResource (preemptive: {isinstance(gpu_trace, simpy.PreemptiveResource)})")
print(f"Condition 1 - starts while both GPUs busy: {over_capacity}")
print(f"Condition 2 - free request started while a paid request waited: {priority_violations}")
print(f"Condition 3 - requests started more than once (interrupted): {restarted}")
print("PASSED" if over_capacity == 0 and priority_violations == 0 and restarted == 0 else "FAILED")

# First 20 events from the start of the 9:00 AM peak (t = 32,400 s)
print("\nFirst 20 events from 9:00 AM:")
print(f"{'Time (s)':>10}  {'Request':<9}{'Tier':<6}{'Event':<8}{'Queue':>6}{'Busy':>6}")
for t, rid, tier, ev, q, busy, _ in [e for e in trace if e[0] >= 32400][:20]:
    print(f"{t:10.2f}  #{rid:<8}{tier:<6}{ev:<8}{q:>6}{busy:>6}")

# One example of a paid request jumping ahead of an earlier free request
arrive_time = {e[1]: e[0] for e in trace if e[3] == "ARRIVE"}
already_started = set()
for e in starts:
    if e[2] == "Paid" and e[0] >= 32400:
        jumped = [f for f in trace if f[3] == "ARRIVE" and f[2] == "Free"
                  and f[0] < arrive_time[e[1]] and f[1] not in already_started]
        if jumped:
            print(f"\nExample: request #{e[1]} (Paid, arrived {arrive_time[e[1]]:.2f}s) started at {e[0]:.2f}s, "
                  f"ahead of request #{jumped[0][1]} (Free, arrived earlier at {jumped[0][0]:.2f}s)")
            break
    already_started.add(e[1])

=== CHECK 2: Event Trace (Alternative 1, seed 42) ===
Resource type: PriorityResource (preemptive: False)
Condition 1 - starts while both GPUs busy: 0
Condition 2 - free request started while a paid request waited: 0
Condition 3 - requests started more than once (interrupted): 0
PASSED

First 20 events from 9:00 AM:
  Time (s)  Request  Tier  Event    Queue  Busy
  32400.26  #1687    Free  ARRIVE       0     2
  32400.26  #1687    Free  START        0     2
  32400.49  #1688    Free  ARRIVE       1     2
  32414.35  #1689    Free  ARRIVE       2     2
  32418.05  #1690    Free  ARRIVE       3     2
  32429.96  #1686    Free  DONE         3     1
  32429.96  #1688    Free  START        2     2
  32433.05  #1691    Free  ARRIVE       3     2
  32435.88  #1692    Paid  ARRIVE       4     2
  32439.89  #1693    Free  ARRIVE       5     2
  32441.92  #1694    Free  ARRIVE       6     2
  32445.28  #1688    Free  DONE         6     1
  32445.28  #1692    Paid  START        5     2
  32445.48

## Scenario 3: Alternative 2 (3 GPUs, FCFS)
**The Concept:** This represents our hardware-based solution. We abandon the VIP lane concept and return to a strict First-Come, First-Served line, but we purchase a 3rd GPU to increase our overall processing bandwidth.

**How the Code Works:** We revert to the standard `simpy.Resource` used in the Baseline, meaning no one gets to cut the line anymore. However, we hardcode the resource `capacity=3`. The logic remains exactly the same as Scenario 1, but the bottleneck is physically wider. Python will now process up to 3 users simultaneously before forcing new arrivals to wait in the queue.

In [6]:
# ==========================================
# SCENARIO 3: ALTERNATIVE 2 (3 GPUs, FCFS)
# ==========================================

free_sla_breaches_alt2 = 0
paid_sla_breaches_alt2 = 0
total_free_users_alt2 = 0
total_paid_users_alt2 = 0

free_wait_times_alt2 = []
paid_wait_times_alt2 = []

def request_process_alt2(env, name, tier, gpu_resource):
    global free_sla_breaches_alt2, paid_sla_breaches_alt2
    global total_free_users_alt2, total_paid_users_alt2

    arrival_time = env.now

    with gpu_resource.request() as req:
        yield req

        wait_time = env.now - arrival_time

        if tier == 'Free':
            total_free_users_alt2 += 1
            free_wait_times_alt2.append(wait_time)
            if wait_time > SLA_THRESHOLD:
                free_sla_breaches_alt2 += 1
        else:
            total_paid_users_alt2 += 1
            paid_wait_times_alt2.append(wait_time)
            if wait_time > SLA_THRESHOLD:
                paid_sla_breaches_alt2 += 1

        service_duration = random.expovariate(1.0 / MEAN_SERVICE_TIME)
        yield env.timeout(service_duration)

def arrival_generator_alt2(env, gpu_resource):
    user_count = 0
    while True:
        current_rate = get_current_arrival_rate(env.now)
        interarrival_time = random.expovariate(current_rate / 60.0)
        yield env.timeout(interarrival_time)

        user_count += 1
        tier = 'Paid' if random.random() < PAID_TIER_PROB else 'Free'
        env.process(request_process_alt2(env, f"User_{user_count}", tier, gpu_resource))

## Replication Loop and Results (All 3 Scenarios)
Execution of Baseline, Priority Queue, and 3-GPU scenarios across 30 sequential replications with fixed random seeds for fair comparison.

In [7]:
# ==========================================
# REPLICATION LOOP & RESULTS AGGREGATION
# ==========================================

rep_free_avg_wait_fcfs, rep_paid_avg_wait_fcfs = [], []
rep_free_breach_rate_fcfs, rep_paid_breach_rate_fcfs = [], []

rep_free_avg_wait_priority, rep_paid_avg_wait_priority = [], []
rep_free_breach_rate_priority, rep_paid_breach_rate_priority = [], []

rep_free_avg_wait_alt2, rep_paid_avg_wait_alt2 = [], []
rep_free_breach_rate_alt2, rep_paid_breach_rate_alt2 = [], []

print("Starting simulation study...")

for rep in range(NUM_REPLICATIONS):

    # --- Running Scenario 1: Baseline (2 GPUs) ---
    random.seed(42 + rep)
    np.random.seed(42 + rep)

    env_fcfs = simpy.Environment()
    gpu_resource_fcfs = simpy.Resource(env_fcfs, capacity=NUM_GPUS)
    env_fcfs.process(arrival_generator_fcfs(env_fcfs, gpu_resource_fcfs))

    free_sla_breaches, paid_sla_breaches = 0, 0
    total_free_users, total_paid_users = 0, 0
    free_wait_times, paid_wait_times = [], []

    env_fcfs.run(until=SIM_DURATION)

    rep_free_avg_wait_fcfs.append(np.mean(free_wait_times) if free_wait_times else 0)
    rep_paid_avg_wait_fcfs.append(np.mean(paid_wait_times) if paid_wait_times else 0)
    rep_free_breach_rate_fcfs.append(free_sla_breaches / total_free_users if total_free_users > 0 else 0)
    rep_paid_breach_rate_fcfs.append(paid_sla_breaches / total_paid_users if total_paid_users > 0 else 0)


    # --- Running Scenario 2: Priority Queue (2 GPUs) ---
    random.seed(42 + rep)
    np.random.seed(42 + rep)
    env_priority = simpy.Environment()

    gpu_resource_priority = simpy.PriorityResource(env_priority, capacity=NUM_GPUS)
    env_priority.process(arrival_generator_priority(env_priority, gpu_resource_priority))

    free_sla_breaches_priority, paid_sla_breaches_priority = 0, 0
    total_free_users_priority, total_paid_users_priority = 0, 0
    free_wait_times_priority, paid_wait_times_priority = [], []

    env_priority.run(until=SIM_DURATION)

    rep_free_avg_wait_priority.append(np.mean(free_wait_times_priority) if free_wait_times_priority else 0)
    rep_paid_avg_wait_priority.append(np.mean(paid_wait_times_priority) if paid_wait_times_priority else 0)
    rep_free_breach_rate_priority.append(free_sla_breaches_priority / total_free_users_priority if total_free_users_priority > 0 else 0)
    rep_paid_breach_rate_priority.append(paid_sla_breaches_priority / total_paid_users_priority if total_paid_users_priority > 0 else 0)


    # --- Running Scenario 3: Alternative 2 (3 GPUs, FCFS) ---
    random.seed(42 + rep)
    np.random.seed(42 + rep)
    env_alt2 = simpy.Environment()

    gpu_resource_alt2 = simpy.Resource(env_alt2, capacity=3)
    env_alt2.process(arrival_generator_alt2(env_alt2, gpu_resource_alt2))

    free_sla_breaches_alt2, paid_sla_breaches_alt2 = 0, 0
    total_free_users_alt2, total_paid_users_alt2 = 0, 0
    free_wait_times_alt2, paid_wait_times_alt2 = [], []

    env_alt2.run(until=SIM_DURATION)

    rep_free_avg_wait_alt2.append(np.mean(free_wait_times_alt2) if free_wait_times_alt2 else 0)
    rep_paid_avg_wait_alt2.append(np.mean(paid_wait_times_alt2) if paid_wait_times_alt2 else 0)
    rep_free_breach_rate_alt2.append(free_sla_breaches_alt2 / total_free_users_alt2 if total_free_users_alt2 > 0 else 0)
    rep_paid_breach_rate_alt2.append(paid_sla_breaches_alt2 / total_paid_users_alt2 if total_paid_users_alt2 > 0 else 0)

# ==========================================
# FINAL AGGREGATION & PRINTING
# ==========================================

print("=== Final Baseline (2 GPUs, FCFS) Results ===")
print(f"Average Free Tier Wait Time: {np.mean(rep_free_avg_wait_fcfs):.2f} seconds")
print(f"Average Paid Tier Wait Time: {np.mean(rep_paid_avg_wait_fcfs):.2f} seconds")
print(f"Average Paid Tier SLA Breach Rate: {np.mean(rep_paid_breach_rate_fcfs)*100:.2f}%")

print("\n=== Final Alternative 1 (2 GPUs, Priority) Results ===")
print(f"Average Free Tier Wait Time: {np.mean(rep_free_avg_wait_priority):.2f} seconds")
print(f"Average Paid Tier Wait Time: {np.mean(rep_paid_avg_wait_priority):.2f} seconds")
print(f"Average Paid Tier SLA Breach Rate: {np.mean(rep_paid_breach_rate_priority)*100:.2f}%")

print("\n=== Final Alternative 2 (3 GPUs, FCFS) Results ===")
print(f"Average Free Tier Wait Time: {np.mean(rep_free_avg_wait_alt2):.2f} seconds")
print(f"Average Paid Tier Wait Time: {np.mean(rep_paid_avg_wait_alt2):.2f} seconds")
print(f"Average Paid Tier SLA Breach Rate: {np.mean(rep_paid_breach_rate_alt2)*100:.2f}%")

Starting simulation study...
=== Final Baseline (2 GPUs, FCFS) Results ===
Average Free Tier Wait Time: 6.33 seconds
Average Paid Tier Wait Time: 6.24 seconds
Average Paid Tier SLA Breach Rate: 0.13%

=== Final Alternative 1 (2 GPUs, Priority) Results ===
Average Free Tier Wait Time: 7.33 seconds
Average Paid Tier Wait Time: 2.23 seconds
Average Paid Tier SLA Breach Rate: 0.00%

=== Final Alternative 2 (3 GPUs, FCFS) Results ===
Average Free Tier Wait Time: 0.82 seconds
Average Paid Tier Wait Time: 0.80 seconds
Average Paid Tier SLA Breach Rate: 0.00%


### Check 3: Conservation Law (Verification)

For non-preemptive priority disciplines where all classes share the same service-time distribution, reordering the queue cannot change total delay. The priority rule should only shift waiting time from free-tier to paid-tier requests.

The arrival-weighted average wait under Alternative 1 is compared with the Baseline FCFS wait:

$$\bar{W} = 0.2\,W_{paid} + 0.8\,W_{free}$$

**Expected:** Both weighted averages are approximately equal.

In [8]:
# ==========================================
# CHECK 3: CONSERVATION LAW (Verification)
# ==========================================
w_alt1 = PAID_TIER_PROB * np.mean(rep_paid_avg_wait_priority) + FREE_TIER_PROB * np.mean(rep_free_avg_wait_priority)
w_base = PAID_TIER_PROB * np.mean(rep_paid_avg_wait_fcfs) + FREE_TIER_PROB * np.mean(rep_free_avg_wait_fcfs)

print("=== CHECK 3: Conservation Law ===")
print(f"Alternative 1 weighted average wait: {w_alt1:.2f} s")
print(f"Baseline weighted average wait:      {w_base:.2f} s")
print(f"Difference: {abs(w_alt1 - w_base):.2f} s")
print("PASSED" if abs(w_alt1 - w_base) < 0.5 else "FAILED")

=== CHECK 3: Conservation Law ===
Alternative 1 weighted average wait: 6.31 s
Baseline weighted average wait:      6.31 s
Difference: 0.01 s
PASSED


## Sensitivity Analysis (+20% Traffic Stress Test)
**The Concept:** Simulations must be tested for worst-case scenarios. We need to know what happens if a viral marketing campaign causes our peak traffic to suddenly spike by 20%. Which system will survive, and which will crash?

**How the Code Works:** We dynamically overwrite our global `PEAK_ARRIVAL_RATE` from 8 to 9.6 requests per minute. We then force Python to silently re-run all three scenarios (Baseline, Priority, and 3 GPUs) under this new, heavier traffic load. We can mathematically prove which system architecture is the most resilient against sudden demand shocks by comparing the Paid tier wait times across all three.

In [9]:
# ==========================================
# STRESS TEST: +20% PEAK ARRIVALS
# ==========================================

PEAK_ARRIVAL_RATE = 9.6  # (8 * 1.20)

rep_paid_wait_stress_fcfs = []
rep_paid_wait_stress_priority = []
rep_paid_wait_stress_alt2 = []

rep_free_wait_stress_fcfs = []
rep_free_wait_stress_priority = []
rep_free_wait_stress_alt2 = []

print("\nRunning +20% Traffic Stress Test...")

for rep in range(NUM_REPLICATIONS):

    # Block 1: Testing the Baseline (2 GPUs, FCFS) under stress.
    random.seed(42 + rep)
    np.random.seed(42 + rep)
    env_fcfs = simpy.Environment()
    gpu_resource_fcfs = simpy.Resource(env_fcfs, capacity=NUM_GPUS)
    env_fcfs.process(arrival_generator_fcfs(env_fcfs, gpu_resource_fcfs))

    free_sla_breaches, paid_sla_breaches, total_free_users, total_paid_users = 0, 0, 0, 0
    free_wait_times, paid_wait_times = [], []

    env_fcfs.run(until=SIM_DURATION)
    rep_paid_wait_stress_fcfs.append(np.mean(paid_wait_times) if paid_wait_times else 0)
    rep_free_wait_stress_fcfs.append(np.mean(free_wait_times) if free_wait_times else 0)

    # Block 2: Testing the Priority Queue (2 GPUs, VIP Lane) under stress.
    random.seed(42 + rep)
    np.random.seed(42 + rep)
    env_priority = simpy.Environment()
    gpu_resource_priority = simpy.PriorityResource(env_priority, capacity=NUM_GPUS)
    env_priority.process(arrival_generator_priority(env_priority, gpu_resource_priority))

    free_sla_breaches_priority, paid_sla_breaches_priority, total_free_users_priority, total_paid_users_priority = 0, 0, 0, 0
    free_wait_times_priority, paid_wait_times_priority = [], []

    env_priority.run(until=SIM_DURATION)
    rep_paid_wait_stress_priority.append(np.mean(paid_wait_times_priority) if paid_wait_times_priority else 0)
    rep_free_wait_stress_priority.append(np.mean(free_wait_times_priority) if free_wait_times_priority else 0)

    # Block 3: Testing Alternative 2 (3 GPUs, FCFS) under stress.
    random.seed(42 + rep)
    np.random.seed(42 + rep)
    env_alt2 = simpy.Environment()
    gpu_resource_alt2 = simpy.Resource(env_alt2, capacity=3)
    env_alt2.process(arrival_generator_alt2(env_alt2, gpu_resource_alt2))

    free_sla_breaches_alt2, paid_sla_breaches_alt2, total_free_users_alt2, total_paid_users_alt2 = 0, 0, 0, 0
    free_wait_times_alt2, paid_wait_times_alt2 = [], []

    env_alt2.run(until=SIM_DURATION)
    rep_paid_wait_stress_alt2.append(np.mean(paid_wait_times_alt2) if paid_wait_times_alt2 else 0)
    rep_free_wait_stress_alt2.append(np.mean(free_wait_times_alt2) if free_wait_times_alt2 else 0)


# --- FINAL AGGREGATION & PRINTING ---
print("=== STRESS TEST RESULTS (Paid Tier Average Wait Time) ===")
print(f"Baseline (2 GPUs, FCFS): {np.mean(rep_paid_wait_stress_fcfs):.2f} seconds")
print(f"Alternative 1 (2 GPUs, Priority): {np.mean(rep_paid_wait_stress_priority):.2f} seconds")
print(f"Alternative 2 (3 GPUs, FCFS): {np.mean(rep_paid_wait_stress_alt2):.2f} seconds")

print("\n=== STRESS TEST RESULTS (Free Tier Average Wait Time) ===")
print(f"Baseline (2 GPUs, FCFS): {np.mean(rep_free_wait_stress_fcfs):.2f} seconds")
print(f"Alternative 1 (2 GPUs, Priority): {np.mean(rep_free_wait_stress_priority):.2f} seconds")
print(f"Alternative 2 (3 GPUs, FCFS): {np.mean(rep_free_wait_stress_alt2):.2f} seconds")

PEAK_ARRIVAL_RATE = 8  # restore normal peak rate


Running +20% Traffic Stress Test...
=== STRESS TEST RESULTS (Paid Tier Average Wait Time) ===
Baseline (2 GPUs, FCFS): 20.52 seconds
Alternative 1 (2 GPUs, Priority): 3.17 seconds
Alternative 2 (3 GPUs, FCFS): 1.57 seconds

=== STRESS TEST RESULTS (Free Tier Average Wait Time) ===
Baseline (2 GPUs, FCFS): 20.51 seconds
Alternative 1 (2 GPUs, Priority): 24.80 seconds
Alternative 2 (3 GPUs, FCFS): 1.53 seconds


## 95% Confidence Intervals
Mean ± 95% confidence interval across the 30 replications, computed as x̄ ± t₀.₀₂₅,₂₉ · s/√30. These values are reported in Tables 3 and 4 of the paper.

In [10]:
# ==========================================
# 95% CONFIDENCE INTERVALS (30 replications)
# ==========================================
from scipy import stats

def ci95(x, scale=1):
    x = np.asarray(x, dtype=float) * scale
    half = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return f"{x.mean():.2f} ± {half:.2f}"

print("=== Normal Peak (mean ± 95% CI, n = 30) ===")
print(pd.DataFrame({
    "Baseline (2 GPUs)": [ci95(rep_free_avg_wait_fcfs), ci95(rep_paid_avg_wait_fcfs), ci95(rep_paid_breach_rate_fcfs, 100)],
    "Alt 1 (Priority)":  [ci95(rep_free_avg_wait_priority), ci95(rep_paid_avg_wait_priority), ci95(rep_paid_breach_rate_priority, 100)],
    "Alt 2 (3 GPUs)":    [ci95(rep_free_avg_wait_alt2), ci95(rep_paid_avg_wait_alt2), ci95(rep_paid_breach_rate_alt2, 100)],
}, index=["Free wait (s)", "Paid wait (s)", "Paid SLA breach (%)"]).to_string())

print("\n=== Stress Test +20% Peak (mean ± 95% CI, n = 30) ===")
print(pd.DataFrame({
    "Paid wait (s)": [ci95(rep_paid_wait_stress_fcfs), ci95(rep_paid_wait_stress_priority), ci95(rep_paid_wait_stress_alt2)],
    "Free wait (s)": [ci95(rep_free_wait_stress_fcfs), ci95(rep_free_wait_stress_priority), ci95(rep_free_wait_stress_alt2)],
}, index=["Baseline (2 GPUs)", "Alt 1 (Priority)", "Alt 2 (3 GPUs)"]).to_string())

=== Normal Peak (mean ± 95% CI, n = 30) ===
                    Baseline (2 GPUs) Alt 1 (Priority) Alt 2 (3 GPUs)
Free wait (s)             6.33 ± 0.45      7.33 ± 0.54    0.82 ± 0.05
Paid wait (s)             6.24 ± 0.44      2.23 ± 0.07    0.80 ± 0.06
Paid SLA breach (%)       0.13 ± 0.12      0.00 ± 0.00    0.00 ± 0.00

=== Stress Test +20% Peak (mean ± 95% CI, n = 30) ===
                  Paid wait (s) Free wait (s)
Baseline (2 GPUs)  20.52 ± 2.11  20.51 ± 2.12
Alt 1 (Priority)    3.17 ± 0.10  24.80 ± 2.62
Alt 2 (3 GPUs)      1.57 ± 0.10   1.53 ± 0.09


### Check 4: Comparison Against Queueing Theory (Validation)

Simulated average waits are compared with analytical **M/M/c** predictions:
- **FCFS configurations** (Baseline, Alternative 2) use the **Erlang C** formula.
- **Alternative 1** uses the **non-preemptive priority M/M/c** formula.

Analytical 24-hour averages are computed by taking the steady-state wait for each traffic window and weighting it by expected arrivals: 2,880 peak and 3,240 off-peak requests (3,456 peak requests in the +20% stress test).

**Expected:** Simulated values fall close to theory, slightly below in most cases. Each 3-hour peak window starts from light off-peak conditions, so the queue needs time to build, while the formulas assume steady state.

In [11]:
# ==========================================
# CHECK 4: COMPARISON AGAINST QUEUEING THEORY (Validation)
# ==========================================
from math import factorial

def erlang_c(lam, mu, c):
    """Probability an arriving request has to wait in an M/M/c queue."""
    a, rho = lam / mu, lam / (c * mu)
    top = a**c / (factorial(c) * (1 - rho))
    return top / (sum(a**k / factorial(k) for k in range(c)) + top)

def mmc_waits(lam, c, mu=60 / MEAN_SERVICE_TIME, p_paid=PAID_TIER_PROB):
    """Steady-state waits in seconds: (FCFS, priority paid, priority free)."""
    w0 = erlang_c(lam, mu, c) / (c * mu)
    s1, rho = p_paid * lam / (c * mu), lam / (c * mu)
    return [60 * w for w in (w0 / (1 - rho), w0 / (1 - s1), w0 / ((1 - s1) * (1 - rho)))]

def day_avg(peak, c, i, off=3):
    """24-hour average: each window's wait weighted by its expected arrivals."""
    n_peak, n_off = 6 * 60 * peak, 18 * 60 * off
    return (n_peak * mmc_waits(peak, c)[i] + n_off * mmc_waits(off, c)[i]) / (n_peak + n_off)

rows = [
    ("Baseline (2 GPUs)", "Paid", np.mean(rep_paid_avg_wait_fcfs),     day_avg(8, 2, 0)),
    ("Baseline (2 GPUs)", "Free", np.mean(rep_free_avg_wait_fcfs),     day_avg(8, 2, 0)),
    ("Alt. 1 (Priority)", "Paid", np.mean(rep_paid_avg_wait_priority), day_avg(8, 2, 1)),
    ("Alt. 1 (Priority)", "Free", np.mean(rep_free_avg_wait_priority), day_avg(8, 2, 2)),
    ("Alt. 2 (3 GPUs)",   "Paid", np.mean(rep_paid_avg_wait_alt2),     day_avg(8, 3, 0)),
    ("Alt. 2 (3 GPUs)",   "Free", np.mean(rep_free_avg_wait_alt2),     day_avg(8, 3, 0)),
    ("Stress, Baseline",  "Paid", np.mean(rep_paid_wait_stress_fcfs),     day_avg(9.6, 2, 0)),
    ("Stress, Alt. 1",    "Paid", np.mean(rep_paid_wait_stress_priority), day_avg(9.6, 2, 1)),
    ("Stress, Alt. 2",    "Paid", np.mean(rep_paid_wait_stress_alt2),     day_avg(9.6, 3, 0)),
]
vv_df = pd.DataFrame(rows, columns=["Configuration", "Tier", "Simulated", "Analytical"])
vv_df["Difference_%"] = (vv_df["Simulated"] - vv_df["Analytical"]) / vv_df["Analytical"] * 100

print("=== CHECK 4: Simulated vs Analytical (seconds) ===")
print(vv_df.round(2).to_string(index=False))
print(f"\nLargest difference: {vv_df['Difference_%'].abs().max():.1f}%")

=== CHECK 4: Simulated vs Analytical (seconds) ===
    Configuration Tier  Simulated  Analytical  Difference_%
Baseline (2 GPUs) Paid       6.24        6.50         -3.95
Baseline (2 GPUs) Free       6.33        6.50         -2.58
Alt. 1 (Priority) Paid       2.23        2.25         -0.80
Alt. 1 (Priority) Free       7.33        7.56         -3.13
  Alt. 2 (3 GPUs) Paid       0.80        0.81         -0.48
  Alt. 2 (3 GPUs) Free       0.82        0.81          2.10
 Stress, Baseline Paid      20.52       19.92          2.99
   Stress, Alt. 1 Paid       3.17        3.17         -0.18
   Stress, Alt. 2 Paid       1.57        1.59         -1.49

Largest difference: 4.0%


In [12]:
# ==========================================
# GENERATE CSV EXPORT
# ==========================================

# Organize all the calculated averages into a clean dictionary
data = {
    "Scenario": [
        "Baseline (2 GPUs, FCFS) - Normal Peak",
        "Alternative 1 (2 GPUs, Priority) - Normal Peak",
        "Alternative 2 (3 GPUs, FCFS) - Normal Peak",
        "Baseline (2 GPUs, FCFS) - Stress Test (+20%)",
        "Alternative 1 (2 GPUs, Priority) - Stress Test (+20%)",
        "Alternative 2 (3 GPUs, FCFS) - Stress Test (+20%)"
    ],
    "Free_Tier_Avg_Wait_sec": [
        round(np.mean(rep_free_avg_wait_fcfs), 2),
        round(np.mean(rep_free_avg_wait_priority), 2),
        round(np.mean(rep_free_avg_wait_alt2), 2),
        round(np.mean(rep_free_wait_stress_fcfs), 2),
        round(np.mean(rep_free_wait_stress_priority), 2),
        round(np.mean(rep_free_wait_stress_alt2), 2)
    ],
    "Paid_Tier_Avg_Wait_sec": [
        round(np.mean(rep_paid_avg_wait_fcfs), 2),
        round(np.mean(rep_paid_avg_wait_priority), 2),
        round(np.mean(rep_paid_avg_wait_alt2), 2),
        round(np.mean(rep_paid_wait_stress_fcfs), 2),
        round(np.mean(rep_paid_wait_stress_priority), 2),
        round(np.mean(rep_paid_wait_stress_alt2), 2)
    ],
    "Paid_Tier_SLA_Breach_Rate_%": [
        round(np.mean(rep_paid_breach_rate_fcfs) * 100, 2),
        round(np.mean(rep_paid_breach_rate_priority) * 100, 2),
        round(np.mean(rep_paid_breach_rate_alt2) * 100, 2),
        "N/A",  # SLA breach rate not recorded in the stress test
        "N/A",
        "N/A"
    ]
}

# Convert the dictionary into a pandas DataFrame (spreadsheet format)
results_df = pd.DataFrame(data)

# Export the DataFrame to a CSV file
csv_filename = "Simulation_Results.csv"
results_df.to_csv(csv_filename, index=False)

print(f"\nSuccessfully generated and saved {csv_filename}!")

from google.colab import files
files.download(csv_filename)


Successfully generated and saved Simulation_Results.csv!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>